# Alexandria Ingest Smoke Notebook

This notebook is a manual smoke check for the real ingest path. It uses the configured database and configured provider adapters through the CLI, then inspects persisted state with SQLAlchemy.

It is not the only validation for ingest behavior; the matching pytest command is at the end.


## Prerequisites

Expected setup:
- Postgres/pgvector is running with the project's `ALEXANDRIA_DATABASE__*` settings. For local Docker, use `task deploy`.
- Provider credentials are present in `.env` or the environment, for example `ALEXANDRIA_EMBEDDING__API_KEY` and `ALEXANDRIA_SUMMARIZER__API_KEY` for OpenAI-backed adapters.
- The notebook is run from the repository root or from the `notebooks/` directory.

Expected output note: the CLI cell should print a document id and source key. If infrastructure or credentials are missing, the CLI should fail before the database inspection cell.


## Ingest Through The CLI

This cell calls the public CLI entrypoint. `ALEXANDRIA_INGEST__MAX_LEAF_DOCS=1` is set only for the subprocess so the smoke document makes the target leaf eligible for a `split.check` outbox row.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import os
import subprocess


repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent

source_key = f"notebook:ingest:{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"
name = "Notebook Ingest Smoke"
body = "A real-infrastructure smoke document added through the Alexandria CLI."

env = os.environ.copy()
env["ALEXANDRIA_INGEST__MAX_LEAF_DOCS"] = "1"

cmd = [
    "uv",
    "run",
    "cli",
    "ingest",
    "--name",
    name,
    "--body",
    body,
    "--source-key",
    source_key,
]

result = subprocess.run(cmd, cwd=repo, env=env, text=True, capture_output=True, check=True)
document_id = result.stdout.strip()

print("document_id:", document_id)
print("source_key:", source_key)


## Inspect Persisted State

Expected notes:
- The document row should exist with the CLI source key, body, generated summary, and provider embedding.
- The document should be attached to an active leaf node whose `doc_count` is at least one.
- A `split.check` outbox job should exist for that leaf because the CLI run used `ALEXANDRIA_INGEST__MAX_LEAF_DOCS=1`.


In [ ]:
from sqlalchemy import select

from domain.entity import Document, Job, Node
from domain.values import JobKind
from infrastructure.config import get_settings
from infrastructure.persistence.db import Db


get_settings.cache_clear()
settings = get_settings()
db = Db(settings.database)

with db.session() as session:
    document = session.scalar(select(Document).where(Document.source_key == source_key))
    assert document is not None, f"document with source_key={source_key!r} was not persisted"
    assert str(document.id) == document_id

    leaf = session.get(Node, document.leaf_id)
    assert leaf is not None

    split_job = session.scalar(
        select(Job)
        .where(Job.kind == JobKind.SPLIT_CHECK.value, Job.key == leaf.id)
        .order_by(Job.created_at.desc())
    )
    assert split_job is not None, f"split.check job for leaf {leaf.id} was not persisted"

    print("document")
    print("  id:", document.id)
    print("  source_key:", document.source_key)
    print("  name:", document.name)
    print("  body:", document.body)
    print("  summary:", document.summary)
    print("  embedding dimensions:", len(document.embedding))

    print("leaf")
    print("  id:", leaf.id)
    print("  kind:", leaf.kind)
    print("  status:", leaf.status)
    print("  doc_count:", leaf.doc_count)

    print("outbox")
    print("  id:", split_job.id)
    print("  kind:", split_job.kind)
    print("  key:", split_job.key)
    print("  status:", split_job.status)
    print("  payload:", split_job.payload)


### Expected Smoke Output

- `document_id` should match the persisted document id found by `source_key`.
- `embedding dimensions` should match the configured embedding adapter output dimensions.
- The leaf should be `kind: leaf` and `status: active` unless a worker processed the split immediately.
- The outbox row should have `kind: split.check` and `payload` containing the leaf node id.


### Matching Pytest Smoke Command

Run this from the repository root:

```bash
uv run pytest tests/integration/test_ingest_flow.py -q
```
